# Advanced `argparse` Problems: Mutually Exclusive Options and Complex Numbers

This notebook contains advanced practice problems with full solutions for CLIs that use:

- `argparse.ArgumentParser`
- `add_mutually_exclusive_group()`
- boolean flags such as `--verbose` and `--quiet`
- required typed arguments
- custom parsing and validation for complex numbers
- testable CLI design

The original idea is a script where the user must provide a complex number with `-n`, and may choose either verbose output, quiet output, or neither.

In [1]:
import argparse
import cmath
import io
import contextlib
from dataclasses import dataclass

## Helper: test CLI parsers inside a notebook

In a normal terminal, `argparse` reads from `sys.argv`. In a notebook, it is better to write a function that accepts an explicit `argv` list.

In [2]:
def run_cli(main_func, argv):
    """
    Run a CLI-style main(argv) function and capture stdout, stderr, and SystemExit code.

    Returns:
        dict with keys: code, stdout, stderr
    """
    out = io.StringIO()
    err = io.StringIO()
    try:
        with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
            result = main_func(argv)
        code = 0 if result is None else result
    except SystemExit as exc:
        code = exc.code
    return {
        "code": code,
        "stdout": out.getvalue(),
        "stderr": err.getvalue(),
    }

## Problem 1 — Refactor the script into a testable `main(argv)` function

**Task.**

Write a CLI program that accepts:

- `-n` / `--number`: a required complex number
- `-v` / `--verbose`: verbose output
- `-q` / `--quiet`: quiet output

The flags `--verbose` and `--quiet` must be mutually exclusive.

Your solution must:

1. Put parser construction in `build_parser()`.
2. Put execution logic in `main(argv=None)`.
3. Return exit code `0` instead of relying on top-level script execution.
4. Be easy to test from a notebook.

In [3]:
def build_parser_problem_1():
    parser = argparse.ArgumentParser(
        prog="complex-polar",
        description="Convert a complex number to polar form.",
    )

    group = parser.add_mutually_exclusive_group()
    group.add_argument("-v", "--verbose", action="store_true", help="show detailed output")
    group.add_argument("-q", "--quiet", action="store_true", help="show minimal output")

    parser.add_argument(
        "-n",
        "--number",
        type=complex,
        required=True,
        help="complex number, for example 3+4j",
    )

    return parser


def main_problem_1(argv=None):
    parser = build_parser_problem_1()
    args = parser.parse_args(argv)

    if args.quiet:
        print("quiet mode...")
        print("nothing to see here.")
    elif args.verbose:
        print("verbose mode...")
        print(f"input: {args.number}")
        print(f"re={args.number.real}, im={args.number.imag}")
        print(f"{args.number} = {cmath.polar(args.number)}")
    else:
        print("normal mode...")
        print(f"{args.number} = {cmath.polar(args.number)}")

    return 0


# Demonstration
for argv in [
    ["-q", "-n", "3+4j"],
    ["-v", "-n", "3+4j"],
    ["-n", "3+4j"],
]:
    print("argv:", argv)
    print(run_cli(main_problem_1, argv)["stdout"])

argv: ['-q', '-n', '3+4j']
quiet mode...
nothing to see here.

argv: ['-v', '-n', '3+4j']
verbose mode...
input: (3+4j)
re=3.0, im=4.0
(3+4j) = (5.0, 0.9272952180016122)

argv: ['-n', '3+4j']
normal mode...
(3+4j) = (5.0, 0.9272952180016122)



**Solution notes.**

This design is better than parsing arguments at import time because:

- the parser can be tested independently;
- the program can be imported without immediately reading command-line arguments;
- `main(argv)` can be called from unit tests or notebooks;
- the final script can still use `if __name__ == "__main__": raise SystemExit(main())`.

## Problem 2 — Prove that the mutually exclusive group works

**Task.**

Write notebook tests that verify:

1. `-v -q` fails.
2. `-v` alone works.
3. `-q` alone works.
4. neither `-v` nor `-q` works.
5. missing `-n` fails.

Use the helper `run_cli()`.

In [4]:
def assert_contains(text, expected):
    assert expected in text, f"Expected {expected!r} to appear in {text!r}"


tests_problem_2 = [
    {
        "name": "verbose and quiet together fail",
        "argv": ["-v", "-q", "-n", "3+4j"],
        "expected_code": 2,
        "stderr_contains": "not allowed with argument",
    },
    {
        "name": "verbose works",
        "argv": ["-v", "-n", "3+4j"],
        "expected_code": 0,
        "stdout_contains": "verbose mode",
    },
    {
        "name": "quiet works",
        "argv": ["-q", "-n", "3+4j"],
        "expected_code": 0,
        "stdout_contains": "quiet mode",
    },
    {
        "name": "normal mode works",
        "argv": ["-n", "3+4j"],
        "expected_code": 0,
        "stdout_contains": "normal mode",
    },
    {
        "name": "missing number fails",
        "argv": ["-v"],
        "expected_code": 2,
        "stderr_contains": "required",
    },
]

for test in tests_problem_2:
    result = run_cli(main_problem_1, test["argv"])
    assert result["code"] == test["expected_code"], (test["name"], result)

    if "stdout_contains" in test:
        assert_contains(result["stdout"], test["stdout_contains"])

    if "stderr_contains" in test:
        assert_contains(result["stderr"], test["stderr_contains"])

print("All Problem 2 tests passed.")

All Problem 2 tests passed.


**Solution notes.**

By default, `argparse` uses exit code `2` for command-line usage errors. A mutually exclusive group catches conflicting flags before your application logic runs.

## Problem 3 — Require exactly one output mode

**Task.**

Modify the CLI so the user must choose exactly one of:

- `--verbose`
- `--quiet`
- `--normal`

This is different from the original behavior, where choosing neither was allowed.

The parser should reject commands that omit all three flags.

In [5]:
def build_parser_problem_3():
    parser = argparse.ArgumentParser(
        prog="complex-polar-required-mode",
        description="Convert a complex number to polar form with an explicit output mode.",
    )

    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument("-v", "--verbose", action="store_true", help="show detailed output")
    group.add_argument("-q", "--quiet", action="store_true", help="show minimal output")
    group.add_argument("--normal", action="store_true", help="show standard output")

    parser.add_argument("-n", "--number", type=complex, required=True)

    return parser


def main_problem_3(argv=None):
    parser = build_parser_problem_3()
    args = parser.parse_args(argv)

    if args.quiet:
        print("quiet mode...")
    elif args.verbose:
        print("verbose mode...")
        print(f"{args.number} = {cmath.polar(args.number)}")
    elif args.normal:
        print("normal mode...")
        print(f"{args.number} = {cmath.polar(args.number)}")

    return 0


# Valid: exactly one mode
print(run_cli(main_problem_3, ["--normal", "-n", "3+4j"])["stdout"])

# Invalid: no mode
missing_mode = run_cli(main_problem_3, ["-n", "3+4j"])
print("exit code:", missing_mode["code"])
print(missing_mode["stderr"].splitlines()[-1])

normal mode...
(3+4j) = (5.0, 0.9272952180016122)

exit code: 2
complex-polar-required-mode: error: one of the arguments -v/--verbose -q/--quiet --normal is required


**Solution notes.**

Use `add_mutually_exclusive_group(required=True)` when the user must choose one and only one option from a set.

Use this carefully: forcing an explicit mode can make scripts less convenient. The original version is often friendlier because it allows a default behavior.

## Problem 4 — Replace two booleans with a single `mode` value

**Task.**

Rewrite the parser so the parsed namespace contains one field named `mode`, whose value is one of:

- `"verbose"`
- `"quiet"`
- `"normal"`

The user should still type `-v`, `-q`, or nothing. The default should be `"normal"`.

Avoid writing logic that checks two separate booleans.

In [6]:
def build_parser_problem_4():
    parser = argparse.ArgumentParser(
        prog="complex-polar-mode",
        description="Convert a complex number to polar form.",
    )

    group = parser.add_mutually_exclusive_group()
    group.add_argument(
        "-v",
        "--verbose",
        dest="mode",
        action="store_const",
        const="verbose",
        help="show detailed output",
    )
    group.add_argument(
        "-q",
        "--quiet",
        dest="mode",
        action="store_const",
        const="quiet",
        help="show minimal output",
    )

    parser.set_defaults(mode="normal")
    parser.add_argument("-n", "--number", type=complex, required=True)

    return parser


def main_problem_4(argv=None):
    parser = build_parser_problem_4()
    args = parser.parse_args(argv)

    z = args.number
    r, theta = cmath.polar(z)

    if args.mode == "quiet":
        print(r)
    elif args.mode == "verbose":
        print(f"mode={args.mode}")
        print(f"input={z}")
        print(f"real={z.real}")
        print(f"imag={z.imag}")
        print(f"radius={r}")
        print(f"angle={theta}")
    else:
        print(f"{z} = ({r}, {theta})")

    return 0


for argv in [["-q", "-n", "3+4j"], ["-v", "-n", "3+4j"], ["-n", "3+4j"]]:
    print("argv:", argv)
    print(run_cli(main_problem_4, argv)["stdout"])

argv: ['-q', '-n', '3+4j']
5.0

argv: ['-v', '-n', '3+4j']
mode=verbose
input=(3+4j)
real=3.0
imag=4.0
radius=5.0
angle=0.9272952180016122

argv: ['-n', '3+4j']
(3+4j) = (5.0, 0.9272952180016122)



**Solution notes.**

A single `mode` value is often cleaner than multiple boolean fields. It prevents impossible internal states and makes application logic easier to extend.

## Problem 5 — Build a safer complex-number parser

**Task.**

The built-in `complex` constructor accepts inputs such as `3+4j`, but error messages may be unfriendly.

Create a custom `argparse` type function named `parse_complex()` that:

1. Accepts valid Python-style complex numbers.
2. Converts uppercase `J` to lowercase `j`.
3. Rejects `nan` and `inf`.
4. Raises `argparse.ArgumentTypeError` with a helpful message.

Then use it for `-n`.

In [7]:
import math


def parse_complex(text):
    """
    Parse a finite complex number from command-line text.

    Examples accepted:
        3+4j
        3-4j
        -2j
        5
        (3+4j)
        3+4J
    """
    normalized = text.strip().replace("J", "j")

    try:
        value = complex(normalized)
    except ValueError as exc:
        raise argparse.ArgumentTypeError(
            f"{text!r} is not a valid complex number. Try examples like 3+4j, 3-4j, or -2j."
        ) from exc

    if not (math.isfinite(value.real) and math.isfinite(value.imag)):
        raise argparse.ArgumentTypeError(
            f"{text!r} must be finite; nan and inf are not allowed."
        )

    return value


def build_parser_problem_5():
    parser = argparse.ArgumentParser(prog="safe-complex-polar")
    group = parser.add_mutually_exclusive_group()
    group.add_argument("-v", "--verbose", dest="mode", action="store_const", const="verbose")
    group.add_argument("-q", "--quiet", dest="mode", action="store_const", const="quiet")
    parser.set_defaults(mode="normal")

    parser.add_argument(
        "-n",
        "--number",
        type=parse_complex,
        required=True,
        metavar="Z",
        help="finite complex number, for example 3+4j",
    )
    return parser


def main_problem_5(argv=None):
    parser = build_parser_problem_5()
    args = parser.parse_args(argv)

    z = args.number
    r, theta = cmath.polar(z)

    if args.mode == "quiet":
        print(r)
    elif args.mode == "verbose":
        print(f"input={z}")
        print(f"rectangular: real={z.real}, imag={z.imag}")
        print(f"polar: radius={r}, angle={theta}")
    else:
        print(f"{z} = ({r}, {theta})")

    return 0


for argv in [
    ["-n", "3+4J"],
    ["-n", "nan+1j"],
    ["-n", "not-a-number"],
]:
    result = run_cli(main_problem_5, argv)
    print("argv:", argv)
    print("code:", result["code"])
    print("stdout:", result["stdout"].strip())
    print("stderr last line:", result["stderr"].splitlines()[-1] if result["stderr"] else "")
    print()

argv: ['-n', '3+4J']
code: 0
stdout: (3+4j) = (5.0, 0.9272952180016122)
stderr last line: 

argv: ['-n', 'nan+1j']
code: 2
stdout: 
stderr last line: safe-complex-polar: error: argument -n/--number: 'nan+1j' must be finite; nan and inf are not allowed.

argv: ['-n', 'not-a-number']
code: 2
stdout: 
stderr last line: safe-complex-polar: error: argument -n/--number: 'not-a-number' is not a valid complex number. Try examples like 3+4j, 3-4j, or -2j.



**Solution notes.**

Custom type functions are a best practice when domain-specific validation matters. They keep validation close to parsing and let `argparse` display consistent usage errors.

## Problem 6 — Add a mutually exclusive output format group

**Task.**

Extend the CLI with a second mutually exclusive group for output format:

- `--tuple`: print `(radius, angle)`
- `--json`: print JSON
- `--csv`: print a CSV row

Rules:

1. The mode group still controls detail: `--verbose`, `--quiet`, or default normal.
2. The format group controls machine-readable shape.
3. Only one format can be selected.
4. Default format is `"tuple"`.
5. In quiet mode, print only the radius, ignoring the selected format.

In [8]:
import json


def build_parser_problem_6():
    parser = argparse.ArgumentParser(prog="complex-polar-formats")

    mode_group = parser.add_mutually_exclusive_group()
    mode_group.add_argument("-v", "--verbose", dest="mode", action="store_const", const="verbose")
    mode_group.add_argument("-q", "--quiet", dest="mode", action="store_const", const="quiet")
    parser.set_defaults(mode="normal")

    format_group = parser.add_mutually_exclusive_group()
    format_group.add_argument("--tuple", dest="fmt", action="store_const", const="tuple")
    format_group.add_argument("--json", dest="fmt", action="store_const", const="json")
    format_group.add_argument("--csv", dest="fmt", action="store_const", const="csv")
    parser.set_defaults(fmt="tuple")

    parser.add_argument("-n", "--number", type=parse_complex, required=True, metavar="Z")

    return parser


def format_polar(z, fmt):
    r, theta = cmath.polar(z)

    if fmt == "tuple":
        return f"({r}, {theta})"

    if fmt == "json":
        return json.dumps(
            {
                "input": {"real": z.real, "imag": z.imag},
                "polar": {"radius": r, "angle": theta},
            },
            sort_keys=True,
        )

    if fmt == "csv":
        return f"real,imag,radius,angle\n{z.real},{z.imag},{r},{theta}"

    raise ValueError(f"unknown format: {fmt}")


def main_problem_6(argv=None):
    parser = build_parser_problem_6()
    args = parser.parse_args(argv)

    z = args.number
    r, _ = cmath.polar(z)

    if args.mode == "quiet":
        print(r)
    elif args.mode == "verbose":
        print(f"mode={args.mode}")
        print(f"format={args.fmt}")
        print(f"input={z}")
        print(format_polar(z, args.fmt))
    else:
        print(format_polar(z, args.fmt))

    return 0


examples_problem_6 = [
    ["-n", "3+4j"],
    ["--json", "-n", "3+4j"],
    ["--csv", "-n", "3+4j"],
    ["--json", "--csv", "-n", "3+4j"],
    ["-q", "--json", "-n", "3+4j"],
]

for argv in examples_problem_6:
    result = run_cli(main_problem_6, argv)
    print("argv:", argv)
    print("code:", result["code"])
    print(result["stdout"] or result["stderr"].splitlines()[-1])
    print()

argv: ['-n', '3+4j']
code: 0
(5.0, 0.9272952180016122)


argv: ['--json', '-n', '3+4j']
code: 0
{"input": {"imag": 4.0, "real": 3.0}, "polar": {"angle": 0.9272952180016122, "radius": 5.0}}


argv: ['--csv', '-n', '3+4j']
code: 0
real,imag,radius,angle
3.0,4.0,5.0,0.9272952180016122


argv: ['--json', '--csv', '-n', '3+4j']
code: 2
complex-polar-formats: error: argument --csv: not allowed with argument --json

argv: ['-q', '--json', '-n', '3+4j']
code: 0
5.0




**Solution notes.**

Multiple mutually exclusive groups are allowed. Each group enforces exclusivity only among its own members.

This is good CLI design because mode and output format are separate concerns.